In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

In [3]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
DATA_PATH = Path('../data/raw/dataset.csv')

In [5]:
raw_df = pd.read_csv(DATA_PATH)
raw_df.head()

,Year,Month,Sector,Hydroelectric Power,Geothermal Energy,Solar Energy,Wind Energy,Wood Energy,Waste Energy,"Fuel Ethanol, Excluding Denaturant",Biomass Losses and Co-products,Biomass Energy,Total Renewable Energy,Renewable Diesel Fuel,Other Biofuels,Conventional Hydroelectric Power,Biodiesel
0,1973,1,Commerical,0.00,0.00,0.0,0.0,0.570,0.000,0.0,0.0,0.570,0.570,0.0,0.0,0.000,0.0
1,1973,1,Electric Power,0.00,0.49,0.0,0.0,0.054,0.157,0.0,0.0,0.211,89.223,0.0,0.0,88.522,0.0
2,1973,1,Industrial,1.04,0.00,0.0,0.0,98.933,0.000,0.0,0.0,98.933,99.973,0.0,0.0,0.000,0.0
3,1973,1,Residential,0.00,0.00,0.0,0.0,30.074,0.000,0.0,0.0,0.000,30.074,0.0,0.0,0.000,0.0
4,1973,1,Transportation,0.00,0.00,0.0,0.0,0.000,0.000,0.0,0.0,0.000,0.000,0.0,0.0,0.000,0.0


In [6]:
raw_df.shape

(3065, 17)

### 2.1 Data Grain & Uniqueness Validation
**Objective:** Verify that each observation represents a unique sector-month-year combination and ensure there are no duplicate records in the dataset.

In [8]:
# check for full-row duplicates
duplicate_rows = raw_df.duplicated().sum()

composite_key = ['Year', 'Month', 'Sector']
key_duplicates = raw_df.duplicated(subset=composite_key).sum()

print(f"Total rows: {len(raw_df):,}")
print(f"Full row duplicates: {duplicate_rows}")
print(f"Composite key ({composite_key}) duplicates: {key_duplicates}")

Total rows: 3,065
Full row duplicates: 0
Composite key (['Year', 'Month', 'Sector']) duplicates: 0


### 2.2 Comprehensive Data Quality Audit
**Objective:** Audit column data types, missing values (explicit `NaN`), implicit missingness (zero-inflation), and unique value counts.

In [9]:
def audit_data_health(df: pd.DataFrame) -> pd.DataFrame:
    """Builds a high-level data health and schema audit table."""
    audit = pd.DataFrame({
        'dtype': df.dtypes,
        'non_null_count': df.notna().sum(),
        'null_count': df.isna().sum(),
        'null_pct (%)': (df.isna().mean() * 100).round(2),
        'zero_count': (df == 0).sum(),
        'zero_pct (%)': ((df == 0).mean() * 100).round(2),
        'unique_values': df.nunique(),
        'sample_value': df.iloc[0]
    })
    return audit.sort_values(by='zero_pct (%)', ascending=False)

In [10]:
audit_summary = audit_data_health(raw_df)
audit_summary

,dtype,non_null_count,null_count,null_pct (%),zero_count,zero_pct (%),unique_values,sample_value
Other Biofuels,float64,3065,0,0.0,2944,96.05,115,0.0
Renewable Diesel Fuel,float64,3065,0,0.0,2908,94.88,158,0.0
Biodiesel,float64,3065,0,0.0,2788,90.96,270,0.0
Biomass Losses and Co-products,float64,3065,0,0.0,2548,83.13,405,0.0
Conventional Hydroelectric Power,float64,3065,0,0.0,2452,80.00,613,0.0
Wind Energy,float64,3065,0,0.0,2274,74.19,473,0.0
Hydroelectric Power,float64,3065,0,0.0,2040,66.56,509,0.0
"Fuel Ethanol, Excluding Denaturant",float64,3065,0,0.0,1514,49.40,939,0.0
Waste Energy,float64,3065,0,0.0,1514,49.40,1381,0.0
Solar Energy,float64,3065,0,0.0,1330,43.39,1054,0.0


In [16]:
raw_df.Sector.value_counts()

Sector
Commerical        613
Electric Power    613
Industrial        613
Residential       613
Transportation    613
Name: count, dtype: int64

### 2.3 Categorical Integrity Check: Sector
**Objective:** Inspect the distribution of unique values in categorical features to identify typos, inconsistent casing, or rare categories.

In [22]:
sector_counts = raw_df['Sector'].value_counts(dropna=False).to_frame(name='record_count')
sector_counts['pct_share (%)'] = (sector_counts['record_count'] / len(raw_df) * 100).round(2)
sector_counts

,record_count,pct_share (%)
Sector,,
Commerical,613,20.0
Electric Power,613,20.0
Industrial,613,20.0
Residential,613,20.0
Transportation,613,20.0


### 2.4 Numerical Distribution, Skewness & Boundary Checks
**Objective:** Assess parametric boundaries (min/max), zero-inflation, and distributional skewness for all energy features.

In [23]:
num_cols = raw_df.select_dtypes(include=[np.number]).columns.drop(['Year','Month'])
num_cols

Index(['Hydroelectric Power', 'Geothermal Energy', 'Solar Energy',
       'Wind Energy', 'Wood Energy', 'Waste Energy',
       'Fuel Ethanol, Excluding Denaturant', 'Biomass Losses and Co-products',
       'Biomass Energy', 'Total Renewable Energy', 'Renewable Diesel Fuel',
       'Other Biofuels', 'Conventional Hydroelectric Power', 'Biodiesel'],
      dtype='object')

In [26]:
num_stats = pd.DataFrame({
    'min': raw_df[num_cols].min(),
    'max': raw_df[num_cols].max(),
    'mean': raw_df[num_cols].mean(),
    'median': raw_df[num_cols].median(),
    'skewness':raw_df[num_cols].skew().round(2),
    'std': raw_df[num_cols].std()
})

num_stats['skew_category'] = num_stats['skewness'].apply(
    lambda x: 'Highly Skewed (Right)' if x > 1 else ('Highly skewed (Left)' if x < - 1 else 'Moderate / Symmetric')
)
num_stats

,min,max,mean,median,skewness,std,skew_category
Hydroelectric Power,-0.002,2.047,0.169759,0.000,2.35,0.373819,Highly Skewed (Right)
Geothermal Energy,0.000,5.951,1.146369,0.357,1.23,1.550857,Highly Skewed (Right)
Solar Energy,0.000,64.040,2.015008,0.004,5.37,5.774511,Highly Skewed (Right)
Wind Energy,0.000,157.409,4.282404,0.000,5.04,18.124793,Highly Skewed (Right)
Wood Energy,0.000,183.628,36.644408,12.062,1.14,46.900639,Highly Skewed (Right)
Waste Energy,0.000,32.875,5.820124,0.108,1.12,8.247359,Highly Skewed (Right)
"Fuel Ethanol, Excluding Denaturant",0.000,104.420,6.976648,0.007,3.44,21.911920,Highly Skewed (Right)
Biomass Losses and Co-products,0.000,75.373,4.834706,0.000,3.47,15.601717,Highly Skewed (Right)
Biomass Energy,0.000,233.200,46.285969,9.716,1.25,64.241520,Highly Skewed (Right)
Total Renewable Energy,0.000,308.175,70.872209,50.984,0.76,71.197761,Moderate / Symmetric
